# Mamba — State Space Models

**Paper:** Mamba: Linear-Time Sequence Modeling with Selective State Spaces (Gu & Dao 2023)

## Why SSMs? The Transformer bottleneck

Transformers are O(N²) in sequence length — attention attends to all token pairs.
For very long sequences (books, codebases, hours of audio), this becomes prohibitive.

**SSMs are O(N) — linear in sequence length.** Each token is processed using a compact hidden state, like an RNN but parallelizable during training.

## The SSM Core Idea

A state space model defines a continuous-time system:
```
h'(t) = A·h(t) + B·x(t)      ← hidden state update
y(t)  = C·h(t) + D·x(t)      ← output
```

Discretize to handle sequences:
```
h_t = Ā·h_{t-1} + B̄·x_t    ← discrete recurrence
y_t = C·h_t                  ← output
```

**Key insight in Mamba (Selective SSM):** A, B, C are **input-dependent** (unlike classical SSMs where they're fixed). The model can selectively remember or forget based on content.

## Resources

| Resource | Link |
|---|---|
| Mamba paper (Gu & Dao 2023) | https://arxiv.org/abs/2312.00752 |
| S4 paper — precursor to Mamba (Gu et al. 2021) | https://arxiv.org/abs/2111.00396 |
| Official Mamba GitHub | https://github.com/state-spaces/mamba |
| Annotated Mamba (walkthrough with code) | https://srush.github.io/annotated-mamba/hard.html |
| Mamba-2 paper (2024, improved theory) | https://arxiv.org/abs/2405.21060 |
| Mamba vs Transformer benchmark blog | https://hazyresearch.stanford.edu/blog/2023-12-11-zoology |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class S4Layer(nn.Module):
    """
    Simplified SSM layer — conceptual implementation.
    Shows the core recurrence. Not the full Mamba selective mechanism.
    """
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # SSM parameters — A, B, C matrices
        # A: [d_model, d_state, d_state] — state transition
        # B: [d_model, d_state] — input projection
        # C: [d_model, d_state] — output projection
        self.A_log = nn.Parameter(torch.randn(d_model, d_state))
        self.B     = nn.Parameter(torch.randn(d_model, d_state))
        self.C     = nn.Parameter(torch.randn(d_model, d_state))
        self.D     = nn.Parameter(torch.ones(d_model))   # skip connection

        self.in_proj  = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        Applies SSM recurrence over sequence dimension.
        """
        batch, seq_len, _ = x.shape
        x_in = self.in_proj(x)   # [batch, seq_len, d_model]

        # A must be stable (eigenvalues < 1) — use -exp(A_log) to ensure negative real part
        A = -torch.exp(self.A_log)   # [d_model, d_state]

        # Discretize A and B (zero-order hold)
        # Δ (delta) controls the discretization step — here simplified to 1
        delta = F.softplus(torch.ones(seq_len, self.d_model, device=x.device))
        Abar  = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0))     # [seq, d_model, d_state]
        Bbar  = delta.unsqueeze(-1) * self.B.unsqueeze(0)           # [seq, d_model, d_state]

        # Recurrence: h_t = Ā * h_{t-1} + B̄ * x_t
        h = torch.zeros(batch, self.d_model, self.d_state, device=x.device)
        outputs = []

        for t in range(seq_len):
            xt = x_in[:, t, :]   # [batch, d_model]
            h  = (Abar[t] * h) + (Bbar[t] * xt.unsqueeze(-1))   # [batch, d_model, d_state]
            yt = (h * self.C.unsqueeze(0)).sum(-1)  # [batch, d_model]
            outputs.append(yt)

        y = torch.stack(outputs, dim=1)   # [batch, seq_len, d_model]
        y = y + x_in * self.D.unsqueeze(0).unsqueeze(0)   # skip connection

        return self.out_proj(y)


batch, seq, d_model = 2, 32, 64
ssm_layer = S4Layer(d_model=d_model, d_state=16)
x         = torch.randn(batch, seq, d_model)
out       = ssm_layer(x)
print('S4 layer output shape:', out.shape)   # [2, 32, 64]

## What makes Mamba different from S4

In S4 (Structured State Spaces for Sequences), A, B, C are **fixed** per layer — same for all inputs.

Mamba's **Selective SSM**: B and C are **functions of the input**:
```
B = Linear(x)   ← input-dependent
C = Linear(x)   ← input-dependent
Δ = Linear(x)   ← input-dependent step size
```
This allows the model to selectively retain information based on content — like attention but O(N).

In [ ]:
class MambaBlock(nn.Module):
    """
    Simplified Mamba block with selective SSM.
    Full implementation: https://github.com/state-spaces/mamba
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model  = d_model
        self.d_state  = d_state
        self.d_inner  = int(expand * d_model)   # expanded inner dimension

        # Input projection — expands to 2 * d_inner (for SSM branch and gate branch)
        self.in_proj  = nn.Linear(d_model, 2 * self.d_inner, bias=False)

        # 1D convolution over sequence — local context before SSM
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                   padding=d_conv-1, groups=self.d_inner, bias=True)

        # Selective SSM parameters — B and C are INPUT-DEPENDENT
        self.x_proj   = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)  # Δ, B, C
        self.dt_proj  = nn.Linear(1, self.d_inner, bias=True)                  # Δ projection

        # Fixed A (log parameterization for stability)
        A = torch.arange(1, d_state + 1).float().unsqueeze(0).expand(self.d_inner, -1)
        self.A_log    = nn.Parameter(torch.log(A))
        self.D        = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm     = nn.LayerNorm(d_model)

    def ssm(self, x):
        """Selective SSM — B, C, delta are input-dependent."""
        batch, seq_len, d = x.shape
        A = -torch.exp(self.A_log)   # [d_inner, d_state]

        # Project x to get selective parameters
        x_dbl  = self.x_proj(x)     # [batch, seq, d_state*2 + 1]
        delta, B, C = x_dbl.split([1, self.d_state, self.d_state], dim=-1)
        delta  = F.softplus(self.dt_proj(delta)).squeeze(-1)   # [batch, seq, d_inner]

        # Simplified recurrence (full Mamba uses parallel scan for speed)
        h = torch.zeros(batch, d, self.d_state, device=x.device)
        ys = []
        for t in range(seq_len):
            delta_t = delta[:, t, :].unsqueeze(-1)          # [batch, d_inner, 1]
            B_t     = B[:, t, :].unsqueeze(1)               # [batch, 1, d_state]
            C_t     = C[:, t, :].unsqueeze(1)               # [batch, 1, d_state]
            Abar_t  = torch.exp(delta_t * A.unsqueeze(0))   # [batch, d_inner, d_state]
            Bbar_t  = delta_t * B_t                          # [batch, d_inner, d_state]
            h       = Abar_t * h + Bbar_t * x[:, t, :].unsqueeze(-1)
            yt      = (h * C_t).sum(-1)                      # [batch, d_inner]
            ys.append(yt)
        return torch.stack(ys, dim=1) + x * self.D           # [batch, seq, d_inner]

    def forward(self, x):
        residual = x
        x = self.norm(x)

        # Split into SSM branch and gate branch
        x_and_z = self.in_proj(x)          # [batch, seq, 2*d_inner]
        x1, z   = x_and_z.chunk(2, dim=-1) # each [batch, seq, d_inner]

        # Conv → SSM on x1 branch
        x1 = self.conv1d(x1.transpose(1, 2))[:, :, :x.shape[1]].transpose(1, 2)
        x1 = F.silu(x1)
        y  = self.ssm(x1)

        # Gate with z branch
        y = y * F.silu(z)

        return residual + self.out_proj(y)


batch, seq, d_model = 2, 32, 64
mamba = MambaBlock(d_model=d_model)
x     = torch.randn(batch, seq, d_model)
out   = mamba(x)
print('Mamba block output shape:', out.shape)   # [2, 32, 64]
print('\nTransformer vs Mamba complexity:')
for N in [512, 2048, 8192]:
    print(f'  N={N:5d}: Transformer O(N²)={N*N:>12,}  |  Mamba O(N)={N:>6,}  |  ratio: {N*N//N}x')